In [ ]:
import pandas as pd
import text_lloom.workbench as wb
import os

## raciaised data

In [ ]:
#read in the data
racialised_data = pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/racialised_reclassed_reviews.parquet')
racialised_themes = pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_theme.parquet')
racialised_context = pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_context.parquet')
racialised_approach = pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_approach.parquet')
racialised_connotation = pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_connotation.parquet')
#places
places = pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/places_filtered.parquet')

In [ ]:
#merge the dataframes
racialised_data = racialised_data.merge(racialised_themes[["review_id", "theme"]], on='review_id', how='left')
racialised_data = racialised_data.merge(racialised_context[["review_id", "context"]], on='review_id', how='left')
racialised_data = racialised_data.merge(racialised_approach[["review_id", "approach"]], on='review_id', how='left')
racialised_data = racialised_data.merge(racialised_connotation[["review_id", "connotation"]], on='review_id', how='left')
racialised_data = racialised_data.drop(columns=racialised_data.columns[racialised_data.isnull().all()])
racialised_data = racialised_data.drop(columns=['classification_status', 'classification_error', 
                                                'failure_batch_id', 'failure_root_cause' , 'needs_summary',
                                                'uploaded_pics_links'])

In [ ]:
racialised_data.info()

In [ ]:
#drop null columns or coluns with more than 50% null values
places = places.drop(columns=places.columns[places.isnull().mean() > 0.5])
places = places.drop(columns='retrieved_time')
places.info()

In [ ]:
racialised_data = racialised_data.merge(places, on=['unique_identifier','place_id'], how='left')

In [ ]:
racialised_data.info()
#save the merged dataframe
racialised_data.to_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/racialised_merged_data_01062026.parquet')


In [ ]:
racialised_data.approach.unique()

## Data Cleaning

In [ ]:

    # This is not valid JSON (no commas) so we parse it manually
def parse_list(val):
    if pd.isna(val):
        return []
    # strip brackets, split on whitespace, clean quotes
    val = str(val).strip().strip("[]")
    items = [v.strip().strip("'\"") for v in val.split("'") if v.strip().strip("'\"") != ',']
    #make lower case and strip whitespace
    items = [i.lower().strip() for i in items]
    return [i for i in items if i]  # drop empty strings


In [ ]:
for col in ['theme', 'connotation', 'context', 'approach']:
    racialised_data[col] = racialised_data[col].apply(parse_list)
    racialised_data = racialised_data.explode(col, ignore_index=True)

In [ ]:
racialised_data.theme.unique()

In [ ]:

racialised_data.connotation.unique()

In [ ]:
racialised_data.context.unique()

In [ ]:
racialised_data.approach.unique()

In [ ]:
racialised_data[['theme', 'context', 'approach', 'connotation']].head()

In [ ]:
#make everything lowercase for easier searching
racialised_data['theme'] = racialised_data['theme'].str.lower()
racialised_data['context'] = racialised_data['context'].str.lower()
racialised_data['approach'] = racialised_data['approach'].str.lower()
racialised_data['connotation'] = racialised_data['connotation'].str.lower()

In [ ]:
racialised_data.info()

In [ ]:
racialised_data['trans_review'] = racialised_data['trans_review'].apply(
    lambda x: x.encode('utf-8', errors='surrogatepass').decode('utf-8', errors='ignore') 
    if isinstance(x, str) else x
)

In [ ]:
racialised_data.to_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/racialised_exploded_data_01062026.parquet')

## lloom analysis  

In [ ]:
import os
import text_lloom.workbench as wb
from text_lloom.llm import Model, EmbedModel
from dotenv import load_dotenv

load_dotenv()

openrouter_key = os.environ.get("OPENROUTER_API_KEY")
OPENROUTER_BASE = "https://openrouter.ai/api/v1"

CHEAP_MODEL = "openai/gpt-4o-mini"
SMART_MODEL = "openai/gpt-4o"

In [ ]:
def clean_surrogates(text):
    if not isinstance(text, str):
        return text
    # encode to bytes ignoring surrogates, decode back to clean string
    return text.encode('utf-16', 'surrogatepass').decode('utf-16', 'ignore')


def get_system_prompt(theme):
    return (
        "You are a qualitative researcher analysing racialised experiences at public swimming pools. "
        "You are working with reviews that have been pre-classified as racialised — meaning they reference "
        "ethnicity, culture, religion, nationality, or race in relation to people's behaviour, attitudes, or characteristics at the pool. "
        f"The reviews you are analysing have specifically been labelled under the theme '{theme}'. "
        "Your task is to identify the underlying concepts and patterns that explain *how* and *why* "
        "these reviews are racialised under this theme — not just what words appear, but what social "
        "dynamics, experiences, or tensions are being described. "
        "Always respond with valid JSON only. "
        "Do not include any text, explanation, or markdown formatting outside the JSON object."
    )

def setup_llm_fn(api_key, theme=""):
    from openai import AsyncOpenAI
    client = AsyncOpenAI(api_key=api_key, base_url=OPENROUTER_BASE)
    client._system_prompt = get_system_prompt(theme)  # attach to client
    return client

async def call_llm_fn(model, prompt):
    system_prompt = getattr(model.client, '_system_prompt', get_system_prompt(""))
    res = await model.client.chat.completions.create(
        model=model.name,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    )
    text = clean_surrogates(res.choices[0].message.content if res else None)
    in_tok  = res.usage.prompt_tokens     if res.usage else 0
    out_tok = res.usage.completion_tokens if res.usage else 0
    return text, (in_tok, out_tok)



In [ ]:
##embeddings
def setup_embed_fn(api_key):
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer("all-MiniLM-L6-v2")


def call_embed_fn(model, text_arr):
    embeddings = model.client.encode(text_arr)
    return embeddings.tolist(), (0, 0)

In [ ]:
def create_lloom(df_subset, theme=""):
    df_clean = df_subset.copy()
    df_clean['trans_review'] = df_clean['trans_review'].apply(clean_surrogates)

    # Build model args with theme-aware prompt
    themed_model_args = dict(
        distill_model=Model(
            setup_fn=lambda key: setup_llm_fn(key, theme=theme),
            fn=call_llm_fn,
            name=CHEAP_MODEL,
            cost=[0.15 / 1_000_000, 0.6 / 1_000_000],
            rate_limit=(500, 5),
            context_window=128_000,
            api_key=openrouter_key,
        ),
        cluster_model=EmbedModel(
            setup_fn=setup_embed_fn,
            fn=call_embed_fn,
            name="all-MiniLM-L6-v2",
            cost=0,
            batch_size=256,
            api_key="none",
        ),
        synth_model=Model(
            setup_fn=lambda key: setup_llm_fn(key, theme=theme),
            fn=call_llm_fn,
            name=SMART_MODEL,
            cost=[2.5 / 1_000_000, 10 / 1_000_000],
            rate_limit=(100, 5),
            context_window=128_000,
            api_key=openrouter_key,
        ),
        score_model=Model(
            setup_fn=lambda key: setup_llm_fn(key, theme=theme),
            fn=call_llm_fn,
            name=CHEAP_MODEL,
            cost=[0.15 / 1_000_000, 0.6 / 1_000_000],
            rate_limit=(500, 5),
            context_window=128_000,
            api_key=openrouter_key,
        ),
    )

    return wb.lloom(
        df=df_clean,
        text_col="trans_review",
        id_col="review_id",
        **themed_model_args,
    )

In [ ]:
# ── 1. Filter and sample ──────────────────────────────────────────
theme = "crowd"
df_crowd_500 = (
    racialised_data[racialised_data["theme"] == theme]
    .drop_duplicates(subset="review_id")
    .sample(n=min(500, len(racialised_data[racialised_data["theme"] == theme])), random_state=42)
    .copy()
)
print(f"{theme}: {len(df_crowd_500)} reviews")



In [ ]:
# ── 2. Create instance (cleans surrogates internally) ────────────
l_crowd_500 = create_lloom(df_crowd_500, theme="crowd")

# ── 3. Run gen() ─────────────────────────────────────────────────
import builtins
original_input = builtins.input
builtins.input = lambda _: "y"
await l_crowd_500.gen(
    params={"synth_n_concepts": 5, "filter_n_quotes": 3, "summ_n_bullets": 3},
    n_synth=1, auto_review=False, debug=True,
)
builtins.input = original_input



In [ ]:
l_crowd_500.select()

In [ ]:
l_crowd_500.show_selected()

In [ ]:
import text_lloom.concept_induction as ci
print(ci.__file__)

In [ ]:
import re

# Path from the above
ci_path = ci.__file__

with open(ci_path, 'r', encoding='utf-8') as f:
    source = f.read()

# The line we need to patch — line ~693
old_line = 'row = [doc_id, text, concept_id, concept_name, concept_prompt, ans, rationale, "", concept_seed]'
new_line = 'row = [doc_id, text.encode("utf-16", "surrogatepass").decode("utf-16", "ignore").encode("ascii", "ignore").decode("ascii") if isinstance(text, str) else text, concept_id, concept_name, concept_prompt, ans, rationale, "", concept_seed]'

if old_line in source:
    patched = source.replace(old_line, new_line)
    with open(ci_path, 'w', encoding='utf-8') as f:
        f.write(patched)
    print("✅ Patched successfully")
else:
    print("❌ Line not found — print the source around line 693 to inspect")
    # Inspect manually
    lines = source.splitlines()
    for i, line in enumerate(lines[688:700], start=689):
        print(f"{i}: {line}")

In [ ]:
def clean_lloom_df(l):
    l.in_df[l.doc_col] = l.in_df[l.doc_col].apply(clean_surrogates)
    # also clean df_filtered and df_to_score if they exist
    for attr in ['df_filtered', 'df_to_score', 'df_bullets']:
        df = getattr(l, attr, None)
        if df is not None and l.doc_col in df.columns:
            df[l.doc_col] = df[l.doc_col].apply(clean_surrogates)
    return l
# ── 4. Clean internal df then score ──────────────────────────────
l_crowd_500 = clean_lloom_df(l_crowd_500)

builtins.input = lambda _: "y"
score_crowd_500 = await l_crowd_500.score()
builtins.input = original_input

In [ ]:
# Run this to see all attributes on the lloom object
print([attr for attr in dir(l_crowd_500) if not attr.startswith('__')])

In [ ]:
import text_lloom.concept_induction as ci

# Find the file location
print(ci.__file__)

##### old

In [ ]:
##LLooM instance creation function for subsets
def create_lloom(df_subset):
    # Clean surrogates from the text column before passing to LLooM
    df_clean = df_subset.copy()
    df_clean['trans_review'] = df_clean['trans_review'].apply(clean_surrogates)
    
    l = wb.lloom(
        df=df_clean,
        text_col="trans_review",
        id_col="review_id",
        **model_args,
    )
    return l

In [ ]:
def clean_df(df):
    df = df.copy()
    df['trans_review'] = df['trans_review'].apply(
        lambda x: x.encode('utf-16', errors='surrogatepass')
                   .decode('utf-16', errors='ignore')
                   .encode('utf-8', errors='ignore')
                   .decode('utf-8', errors='ignore')
        if isinstance(x, str) else x
    )
    return df

df_crowd_500 = clean_df(racialised_data)

In [ ]:
# After creating the instance, clean the internal df LLooM holds
def clean_lloom_df(l):
    l.df[l.doc_col] = l.df[l.doc_col].apply(clean_surrogates)
    return l

# Apply it before scoring
l_crowd_500 = clean_lloom_df(l_crowd_500)

original_input = builtins.input
builtins.input = lambda _: "y"
score_crowd_500 = await l_crowd_500.score()
builtins.input = original_input

In [ ]:
theme = "crowd"

df_crowd_500= (
    racialised_data[racialised_data["theme"] == theme]
    .drop_duplicates(subset="review_id")
    .sample(n=min(500, len(racialised_data[racialised_data["theme"] == theme])), random_state=42)
    .copy()
)
df_crowd_500['trans_review'] =df_crowd_500['trans_review'].apply(
    lambda x: x.encode('utf-8', errors='surrogatepass').decode('utf-8', errors='ignore') 
    if isinstance(x, str) else x
)
print(f"{theme}: {len(df_crowd_500)} reviews")

l_crowd_500 = create_lloom(df_crowd_500)

In [ ]:
# # Test AsyncOpenAI → OpenRouter is working correctly
# from openai import AsyncOpenAI
# import os

# client = AsyncOpenAI(
#     api_key=os.environ.get("OPENROUTER_API_KEY"),
#     base_url="https://openrouter.ai/api/v1"
# )

# res = await client.chat.completions.create(
#     model="openai/gpt-4o-mini",
#     temperature=0,
#     messages=[{"role": "user", "content": "Say hello in one word."}]
# )
# print(res.choices[0].message.content)

In [ ]:
# Cancel the current run first (Kernel → Interrupt), then run this
# Force it to skip the confirmation prompt

# Monkey-patch to auto-confirm
import builtins
import time
start = time.time()
original_input = builtins.input
builtins.input = lambda _: "y"

await l_crowd_500.gen(
    params={
        "synth_n_concepts": 3,
        "filter_n_quotes": 2,
        "summ_n_bullets": 2,
    },
    n_synth=1,
    auto_review=True,
    debug=True,
)

builtins.input = original_input  # restore after

In [ ]:
l_crowd_500.select()

In [ ]:
l_crowd_500.show_selected()

In [ ]:
l_crowd_500.summary()

In [ ]:
original_input = builtins.input
builtins.input = lambda _: "y"
score_crowd_500 = await l_crowd_500.score()
builtins.input = original_input  # restore after

In [ ]:
##Run all themes 
lloom_results = {}

for theme in sorted(
    racialised_data["theme"].dropna().unique()
):

    df_subset = (
        racialised_data[
            racialised_data["theme"] == theme
        ]
        .drop_duplicates(subset="review_id")
        .copy()
    )

    if len(df_subset) < 5:
        print(f"Skipping {theme}: {len(df_subset)} reviews")
        continue

    print(f"Running {theme}: {len(df_subset)} reviews")

    l = create_lloom(df_subset)

    await l.gen()

    lloom_results[theme] = l

In [ ]:
racialised_data.connotation.unique()